# LTFSR-Meta - Run an experiment on CIFAR-100-LT

This is the **single notebook** you run (locally or on Kaggle). It only sets the
configuration and calls the modules in `src/` - no training logic lives here.

Pipeline: **configure -> load data -> train one method -> evaluate -> visualise.**

Thesis of the study: *on long-tail data the bottleneck is the classifier, not the
features.* Each method fixes the imbalance at a different level, so the
**balanced / few-shot accuracy** rises step by step:

| METHOD | Fix at level | Idea |
|---|---|---|
| `baseline` | - | ResNet + cross-entropy (head-biased reference) |
| `balanced_softmax` | loss | add the class-prior to the logits while training |
| `decoupling` | classifier | train features, then re-train the head class-balanced (cRT) |
| `supcon` | representation | SupCon features + cRT classifier (**best**) |
| `meta` | (bonus) | episodic ProtoNet, reported on the few-shot N-way axis |

Compare runs with the table cell near the end. Headline metric = **balanced_accuracy** and **few_shot_accuracy**, not plain top-1.

## 1. Make `src/` importable (works locally and on Kaggle)

In [ ]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    """Return the folder containing `src/` (search cwd, parents, Kaggle inputs)."""
    candidates = [Path.cwd(), *Path.cwd().parents]
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if base.exists():
            candidates += [p for p in base.iterdir() if p.is_dir()]
    for path in candidates:
        if (path / "src" / "__init__.py").exists():
            return path
    return Path.cwd()


PROJECT_DIR = find_project_root()
sys.path.insert(0, str(PROJECT_DIR))
print("Project root:", PROJECT_DIR)

## 2. Configuration — the only cell you normally edit

On Kaggle, point `DATA_DIR` at your uploaded dataset (e.g.
`/kaggle/input/cifar-100-lt/CIFAR-100-LT`) and `OUTPUT_DIR` at `/kaggle/working`.


In [ ]:
# --- paths ---
DATA_DIR = PROJECT_DIR / "data" / "CIFAR-100-LT"   # folder with train/ test/ class_counts.json
OUTPUT_DIR = PROJECT_DIR / "outputs"               # where run folders are written
BUILD_DATASET = False   # set True on Kaggle to build CIFAR-100-LT into OUTPUT_DIR first

# --- method: "baseline" | "balanced_softmax" | "decoupling" | "supcon" | "meta" ---
METHOD = "baseline"

# --- setup: MAIN (from scratch) vs optional PRETRAINED reference table ---
# PRETRAINED=False -> CIFAR stem, train from scratch on 32x32  (MAIN; keep IMAGE_SIZE=32)
# PRETRAINED=True  -> ImageNet ResNet-18; you MUST set IMAGE_SIZE=224 (optional table only)
PRETRAINED = False
IMAGE_SIZE = 32

# --- core hyperparameters ---
NUM_CLASSES = 100
BATCH_SIZE = 128
LEARNING_RATE = 0.1     # SGD + cosine for all classifier-style methods
EPOCHS = 200            # from-scratch CIFAR needs a long schedule for good results
SEED = 42
NUM_WORKERS = 2
DEVICE = None           # None = auto (CUDA if available), or "cpu" / "cuda"

# --- quick smoke test (None = use the full dataset) ---
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None

# --- decoupling / cRT classifier-retraining stage (Methods 3 & 4) ---
CRT_EPOCHS = 10
CRT_LR = 0.1

# --- SupCon contrastive pre-training (Method 4) ---
PRETRAIN_EPOCHS = 200
PRETRAIN_LR = 0.5
TEMPERATURE = 0.07

# --- few-shot / meta-learning (Method 5, bonus) ---
N_WAY = 5
K_SHOT = 5
N_QUERY = 15
EPISODES_PER_EPOCH = 100
META_LR = 0.001

## 3. Imports, reproducibility, device, run folder

In [ ]:
import random

import numpy as np
import torch

from src.datasets.cifar_lt import (build_transforms, load_class_counts, load_split,
                                    make_loader, split_shot_groups)
from src.evaluation import visualize
from src.evaluation.metrics import compute_metrics, format_metrics
from src.trainers.engine import evaluate
from src.utils.experiment import create_run_dir, save_config, save_history, save_metrics
from src.utils.seed import resolve_device, set_seed

set_seed(SEED)
device = resolve_device(DEVICE)
run_dir = create_run_dir(OUTPUT_DIR, run_name=METHOD)
print("Device:", device, "| Run dir:", run_dir)

## 4. (Optional) Build the dataset, then load it

In [ ]:
if BUILD_DATASET:
    import subprocess
    subprocess.run([sys.executable, str(PROJECT_DIR / "data" / "prepare_datasets.py"),
                    "--dataset", "cifar100-lt",
                    "--data_dir", str(OUTPUT_DIR), "--overwrite"], check=True)
    DATA_DIR = OUTPUT_DIR / "CIFAR-100-LT"

assert (DATA_DIR / "train").exists(), f"No train/ folder under {DATA_DIR}"
class_counts = load_class_counts(DATA_DIR)
shot_groups = split_shot_groups(class_counts)
print("Classes per group:", {name: len(ids) for name, ids in shot_groups.items()})
print("Head class count:", max(class_counts.values()), "| Tail class count:", min(class_counts.values()))

In [ ]:
def subsample(dataset, max_samples, seed=SEED):
    """Keep a random subset of an ImageFolder in place (smoke testing only).

    Editing `.samples`/`.targets` (instead of wrapping in Subset) keeps the
    attributes the episodic meta-trainer and the balanced sampler rely on.
    """
    if max_samples is None or max_samples >= len(dataset.samples):
        return dataset
    rng = random.Random(seed)
    keep = rng.sample(range(len(dataset.samples)), max_samples)
    dataset.samples = [dataset.samples[i] for i in keep]
    dataset.targets = [dataset.targets[i] for i in keep]
    return dataset


# Three views of the data: augmented train (for learning), clean train (for
# prototypes / t-SNE), and the test set (evaluation, the standard LT protocol).
train_aug = subsample(load_split(DATA_DIR, "train", build_transforms(train=True, image_size=IMAGE_SIZE)), MAX_TRAIN_SAMPLES)
train_eval = subsample(load_split(DATA_DIR, "train", build_transforms(train=False, image_size=IMAGE_SIZE)), MAX_TRAIN_SAMPLES)
test_eval = subsample(load_split(DATA_DIR, "test", build_transforms(train=False, image_size=IMAGE_SIZE)), MAX_TEST_SAMPLES)

train_loader = make_loader(train_aug, BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
train_eval_loader = make_loader(train_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = make_loader(test_eval, BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
print(f"train={len(train_aug)} images | test={len(test_eval)} images")

> **Note on evaluation.** CIFAR-100-LT is trained on the long-tailed split and
> evaluated on the *balanced* test set — that is the standard benchmark protocol,
> so we monitor and report on the test set directly (no separate validation
> split). For a stricter setup you could hold out a validation split here.

## 5. Look at the data (long-tail profile)

In [ ]:
visualize.plot_class_frequency(class_counts, run_dir)
visualize.plot_shot_distribution(shot_groups, run_dir)
print("Saved class_frequency.png and shot_distribution.png to", run_dir)

## 6. Train the chosen method

In [ ]:
pretrain_history = None

if METHOD == "baseline":
    from src.trainers.baseline_trainer import train_baseline
    model, history = train_baseline(train_loader, test_loader, NUM_CLASSES, device, run_dir,
                                     epochs=EPOCHS, learning_rate=LEARNING_RATE, pretrained=PRETRAINED)
    result = evaluate(model, test_loader, device, collect_features=True)

elif METHOD == "balanced_softmax":
    from src.trainers.baseline_trainer import train_baseline
    from src.trainers.losses import BalancedSoftmaxLoss
    counts = [class_counts[c] for c in range(NUM_CLASSES)]
    model, history = train_baseline(train_loader, test_loader, NUM_CLASSES, device, run_dir,
                                     epochs=EPOCHS, learning_rate=LEARNING_RATE, pretrained=PRETRAINED,
                                     criterion=BalancedSoftmaxLoss(counts))
    result = evaluate(model, test_loader, device, collect_features=True)

elif METHOD == "decoupling":
    from src.trainers.decoupling_trainer import train_decoupling
    model, history = train_decoupling(train_loader, train_aug, test_loader, NUM_CLASSES, device, run_dir,
                                      epochs=EPOCHS, learning_rate=LEARNING_RATE,
                                      crt_epochs=CRT_EPOCHS, crt_lr=CRT_LR,
                                      batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pretrained=PRETRAINED)
    result = evaluate(model, test_loader, device, collect_features=True)

elif METHOD == "supcon":
    from src.trainers.contrastive_trainer import train_contrastive
    model, history, pretrain_history = train_contrastive(
        DATA_DIR, train_aug, test_loader, NUM_CLASSES, device, run_dir,
        pretrain_epochs=PRETRAIN_EPOCHS, probe_epochs=CRT_EPOCHS,
        batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
        pretrain_lr=PRETRAIN_LR, probe_lr=CRT_LR,
        temperature=TEMPERATURE, pretrained=PRETRAINED, image_size=IMAGE_SIZE)
    result = evaluate(model, test_loader, device, collect_features=True)

elif METHOD == "meta":
    from src.trainers.meta_trainer import evaluate_meta, train_meta
    encoder, history = train_meta(train_aug, train_eval, device, run_dir,
                                  epochs=EPOCHS, episodes_per_epoch=EPISODES_PER_EPOCH,
                                  n_way=N_WAY, k_shot=K_SHOT, n_query=N_QUERY,
                                  learning_rate=META_LR, pretrained=PRETRAINED, seed=SEED)
    result = evaluate_meta(encoder, train_eval_loader, test_loader, NUM_CLASSES, device)

else:
    raise ValueError(f"Unknown METHOD: {METHOD!r}")

print("Training finished. Test accuracy:", round(result["accuracy"], 4))

## 7. Metrics + save everything for this run

In [ ]:
metrics = compute_metrics(result["y_true"], result["y_pred"], NUM_CLASSES, shot_groups)
print(format_metrics(metrics))

config = {k: v for k, v in dict(
    method=METHOD, num_classes=NUM_CLASSES, batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE, epochs=EPOCHS, seed=SEED,
    pretrained=PRETRAINED, image_size=IMAGE_SIZE,
    device=str(device), data_dir=str(DATA_DIR),
    crt_epochs=CRT_EPOCHS, crt_lr=CRT_LR,
    pretrain_epochs=PRETRAIN_EPOCHS, pretrain_lr=PRETRAIN_LR, temperature=TEMPERATURE,
    n_way=N_WAY, k_shot=K_SHOT, n_query=N_QUERY, episodes_per_epoch=EPISODES_PER_EPOCH,
).items()}

save_config(run_dir, config)
save_metrics(run_dir, metrics)
save_history(run_dir, history)
if pretrain_history is not None:
    pretrain_history.to_csv(run_dir / "pretrain_metrics.csv", index=False)
print("Saved config.json, metrics.json, metrics.csv to", run_dir)

## 8. Visualise the results

In [ ]:
visualize.plot_training_curves(history, run_dir)
visualize.plot_confusion_matrices(result["y_true"], result["y_pred"], NUM_CLASSES, run_dir)
if "features" in result:
    visualize.plot_tsne(result["features"], result["y_true"], run_dir)

print("Figures saved to", run_dir)
print(sorted(p.name for p in run_dir.glob("*.png")))

## 9. Compare all methods (headline table for the report)

Run after training several methods. Gathers every `outputs/<method>/metrics.json` into one table sorted by balanced accuracy.

In [ ]:
from src.utils.experiment import compare_runs

comparison = compare_runs(OUTPUT_DIR)
comparison.to_csv(OUTPUT_DIR / "comparison.csv", index=False)
comparison

## 9. Summary

The run folder now contains a complete, reproducible record:

```
outputs/<method>/
├── config.json                      # exact hyperparameters
├── metrics.csv                      # per-epoch history
├── metrics.json                     # final test metrics (incl. many/med/few, g-mean, mcc)
├── best_model.pt                    # best checkpoint
├── curve_loss.png / curve_accuracy.png
├── confusion_matrix.png / confusion_matrix_normalized.png
├── class_frequency.png / shot_distribution.png
└── tsne.png
```

To **compare methods**, change `METHOD` in cell 2 and Run All again — each method
writes to its own folder, so nothing is overwritten. Then compare `g_mean`,
`balanced_accuracy`, and `few_shot_accuracy` across `outputs/*/metrics.json`.
